In [1]:
# -----------------------------
# Indlæsning af data
# -----------------------------

import pandas as pd
import numpy as np

df = pd.read_csv("Data/2026_lolmatches.csv", low_memory=False)

# Behold kun komplette games
df = df[df["datacompleteness"] == "complete"]

# Fjern team rows (vi arbejder stadig på player rows)
df = df[df["position"] != "team"]

# Konverter dato
df["date"] = pd.to_datetime(df["date"])

# Sortér kronologisk (KRITISK for no leakage)
df = df.sort_values("date").reset_index(drop=True)

In [2]:
# -----------------------------
# Helper funktion
# -----------------------------

def expected(r1, r2):
    # Klassisk Elo win probability
    return 1 / (1 + 10 ** ((r2 - r1) / 400))

In [3]:
# -----------------------------
# Elo setup
# -----------------------------

# Start-rating for alle hold
BASE_RATING = 1500

# K-faktor (hvor meget ratings ændrer sig)
K = 20

# Gemmer Elo-rating for hvert hold
team_ratings = {}

In [4]:
# -----------------------------
# Elo beregning
# -----------------------------

# Liste til at gemme data (features)
rows = []

# Loop over hver kamp (groupby gameid)
for game_id, game in df.groupby("gameid"):

    # Skip hvis ikke 5 spillere pr. hold
    if len(game) != 10:
        continue

    # Find de to hold i kampen
    teams = game["teamname"].unique()
    if len(teams) != 2:
        continue

    t1, t2 = teams

    # Hent Elo ratings (eller brug startværdi)
    r1 = team_ratings.get(t1, BASE_RATING)
    r2 = team_ratings.get(t2, BASE_RATING)

    # Beregn win probability for t1
    p1 = expected(r1, r2)

    # Faktisk resultat for t1 (1 = win, 0 = loss)
    result1 = game[game["teamname"] == t1]["result"].iloc[0]

    # -----------------------------
    # Gem features (før update)
    # -----------------------------

    rows.append({
        "gameid": game_id,
        "team_blue": t1,
        "team_red": t2,
        "elo_blue": r1,
        "elo_red": r2,
        "elo_diff": r1 - r2,
        "result": result1
    })

    # -----------------------------
    # Opdater ratings
    # -----------------------------

    # Opdater Elo for begge hold efter kampen
    team_ratings[t1] = r1 + K * (result1 - p1)
    team_ratings[t2] = r2 + K * ((1 - result1) - (1 - p1))

In [5]:
# -----------------------------
# Klargøring af modeldata
# -----------------------------

# Lav DataFrame fra gemte features
matches = pd.DataFrame(rows)

# Target-variabel (det vi vil forudsige)
# 1 = Blue vinder, 0 = Blue taber
matches["result"] = matches["result"]

In [6]:
# -----------------------------
# Feature engineering
# -----------------------------

# Data er fra Blue-side perspektiv
matches["blue_side"] = 1

# Elo difference (styrkeforskel mellem hold)
matches["elo_diff"] = matches["elo_blue"] - matches["elo_red"]

In [7]:
# -----------------------------
# Udvælg features til modellen
# -----------------------------

# Vælg de features og target vi vil bruge
model_data = matches[[
    "elo_diff",   # forskel i Elo
    "blue_side",  # side (altid 1 her)
    "result"      # target (win/loss)
]]

# Vis de første rækker
print(model_data.head())

   elo_diff  blue_side  result
0       0.0          1       1
1       0.0          1       1
2       0.0          1       1
3       0.0          1       0
4      10.0          1       1


In [8]:
# -----------------------------
# Elo model
# -----------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

# Feature (input) og target (output)
X = model_data[["elo_diff"]]
y = model_data["result"]

# Opret og træn model
model = LogisticRegression()
model.fit(X, y)

# Forudsig sandsynligheder for win
probs = model.predict_proba(X)[:, 1]

# Lav klassifikation (0/1)
preds = (probs > 0.5).astype(int)

# Evaluer model
auc = roc_auc_score(y, probs)
acc = accuracy_score(y, preds)
loss = log_loss(y, probs)

# Print resultater
print("AUC:", auc)
print("Accuracy:", acc)
print("Log Loss:", loss)

AUC: 0.63919655789384
Accuracy: 0.5954785954785955
Log Loss: 0.6623685462837478


In [9]:
# Lav en kopi af datasættet
elo_df = matches.copy()

# Gem som CSV fil (uden index kolonne) som bruges tl Samlet model
elo_df.to_csv("Data/elo_features.csv", index=False)